# 1. Exploratory Data Analysis (EDA) - Works on Any Dataset
**Steps:** Load data -> Overview -> Missing values -> Descriptive stats -> Histograms -> Boxplots/Outliers -> Categorical counts -> Correlation heatmap -> Pairplot

> Only change the **CONFIG** cell. Everything else adapts automatically.

## Step 0: Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (8, 5)
pd.set_option('display.max_columns', None)

## Step 1: CONFIG + Load Dataset

In [ ]:
# ---------------- CONFIG (edit only this) ----------------
FILE_PATH = None        # e.g. 'data.csv' or 'data.xlsx'. Keep None to use a sample dataset
TARGET    = None        # optional: name of target column (e.g. 'species'), else None
# ----------------------------------------------------------

if FILE_PATH is None:
    # Sample dataset so the notebook runs out-of-the-box
    from sklearn.datasets import load_iris
    iris = load_iris(as_frame=True)
    df = iris.frame.copy()
    df['species'] = df['target'].map(dict(enumerate(iris.target_names)))
    df = df.drop(columns='target')
    TARGET = 'species'
elif FILE_PATH.endswith('.csv'):
    df = pd.read_csv(FILE_PATH)
elif FILE_PATH.endswith(('.xlsx', '.xls')):
    df = pd.read_excel(FILE_PATH)
else:
    raise ValueError('Unsupported file type')

df.head()

## Step 2: Dataset Overview

In [ ]:
print('Shape (rows, columns):', df.shape)
print('\nColumn names:', list(df.columns))
print('\nData types & non-null counts:')
df.info()

In [ ]:
# Auto-detect numeric and categorical columns
num_cols = df.select_dtypes(include=np.number).columns.tolist()
cat_cols = df.select_dtypes(exclude=np.number).columns.tolist()

print('Numeric columns    :', num_cols)
print('Categorical columns:', cat_cols)
print('Duplicate rows     :', df.duplicated().sum())

## Step 3: Missing Values

In [ ]:
missing = pd.DataFrame({
    'missing_count': df.isnull().sum(),
    'missing_%': (df.isnull().mean() * 100).round(2)
})
print(missing[missing['missing_count'] > 0] if missing['missing_count'].sum() > 0 else 'No missing values')

# Heatmap of missing values (only if any exist)
if df.isnull().sum().sum() > 0:
    plt.figure(figsize=(10, 4))
    sns.heatmap(df.isnull(), cbar=False, yticklabels=False, cmap='viridis')
    plt.title('Missing Values Map')
    plt.show()

## Step 4: Descriptive Statistics

In [ ]:
# Mean, std, min, quartiles, max for numeric columns
df.describe().T

In [ ]:
# Extended stats: median, mode, variance, range, IQR, skewness, kurtosis
stats = pd.DataFrame({
    'mean'    : df[num_cols].mean(),
    'median'  : df[num_cols].median(),
    'mode'    : df[num_cols].mode().iloc[0],
    'std'     : df[num_cols].std(),
    'variance': df[num_cols].var(),
    'min'     : df[num_cols].min(),
    'max'     : df[num_cols].max(),
    'range'   : df[num_cols].max() - df[num_cols].min(),
    'IQR'     : df[num_cols].quantile(0.75) - df[num_cols].quantile(0.25),
    'skewness': df[num_cols].skew(),
    'kurtosis': df[num_cols].kurt()
})
stats.round(3)

In [ ]:
# Summary of categorical columns
if cat_cols:
    display(df[cat_cols].describe().T)
else:
    print('No categorical columns')

## Step 5: Histograms (with KDE curve)

In [ ]:
def plot_grid(n, ncols=3, h=4):
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, h*nrows))
    axes = np.array(axes).reshape(-1)
    return fig, axes

fig, axes = plot_grid(len(num_cols))
for ax, col in zip(axes, num_cols):
    sns.histplot(df[col].dropna(), kde=True, bins=30, ax=ax, color='steelblue')
    ax.axvline(df[col].mean(),   color='red',   linestyle='--', label='Mean')
    ax.axvline(df[col].median(), color='green', linestyle='-.', label='Median')
    ax.set_title(f'{col}  (skew={df[col].skew():.2f})')
    ax.legend()
for ax in axes[len(num_cols):]:
    ax.axis('off')
plt.tight_layout()
plt.show()

## Step 6: Boxplots & Outlier Detection (IQR method)

In [ ]:
fig, axes = plot_grid(len(num_cols))
for ax, col in zip(axes, num_cols):
    sns.boxplot(y=df[col], ax=ax, color='lightcoral')
    ax.set_title(col)
for ax in axes[len(num_cols):]:
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Count outliers per column using IQR rule: outside [Q1 - 1.5*IQR, Q3 + 1.5*IQR]
out = {}
for col in num_cols:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    low, high = q1 - 1.5*iqr, q3 + 1.5*iqr
    out[col] = ((df[col] < low) | (df[col] > high)).sum()
pd.DataFrame({'outlier_count': out})

## Step 7: Categorical Variables (Count Plots)

In [ ]:
if cat_cols:
    fig, axes = plot_grid(len(cat_cols))
    for ax, col in zip(axes, cat_cols):
        if df[col].nunique() <= 20:            # skip high-cardinality columns
            order = df[col].value_counts().index
            sns.countplot(x=df[col], order=order, ax=ax, palette='Set2')
            ax.set_title(f'Count of {col}')
            ax.tick_params(axis='x', rotation=45)
        else:
            ax.set_title(f'{col}: {df[col].nunique()} unique (skipped)')
            ax.axis('off')
    for ax in axes[len(cat_cols):]:
        ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('No categorical columns')

## Step 8: Correlation Matrix & Heatmap

In [ ]:
corr = df[num_cols].corr()          # Pearson by default; use method='spearman' for rank-based
corr.round(2)

In [ ]:
plt.figure(figsize=(max(7, len(num_cols)), max(6, len(num_cols)*0.8)))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
# Triangle (masked) heatmap - cleaner for many variables
mask = np.triu(np.ones_like(corr, dtype=bool))
plt.figure(figsize=(max(7, len(num_cols)), max(6, len(num_cols)*0.8)))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            vmin=-1, vmax=1, square=True, linewidths=0.5)
plt.title('Correlation Heatmap (Lower Triangle)')
plt.show()

In [ ]:
# Strongly correlated pairs (|r| >= 0.7) - useful to mention multicollinearity
pairs = corr.where(np.tril(np.ones(corr.shape), k=-1).astype(bool)).stack().reset_index()
pairs.columns = ['Feature 1', 'Feature 2', 'Correlation']
strong = pairs[pairs['Correlation'].abs() >= 0.7].sort_values('Correlation', key=abs, ascending=False)
print('Strongly correlated pairs (|r| >= 0.7):')
strong if len(strong) else print('None')

In [ ]:
# Correlation of every feature with the target (only if TARGET is numeric)
if TARGET and TARGET in num_cols:
    print(corr[TARGET].drop(TARGET).sort_values(key=abs, ascending=False).round(3))
else:
    print('TARGET is None or not numeric - skipped')

## Step 9: Pairplot & Group Comparison

In [ ]:
# Pairplot (limit to first 5 numeric columns so it runs fast)
cols_for_pair = num_cols[:5]
if TARGET and TARGET in cat_cols and df[TARGET].nunique() <= 10:
    sns.pairplot(df[cols_for_pair + [TARGET]], hue=TARGET, diag_kind='kde', corner=True)
else:
    sns.pairplot(df[cols_for_pair], diag_kind='kde', corner=True)
plt.show()

In [ ]:
# Group-wise mean of numeric columns by a categorical target
if TARGET and TARGET in cat_cols:
    display(df.groupby(TARGET)[num_cols].agg(['mean', 'std']).round(3))
    fig, axes = plot_grid(len(num_cols))
    for ax, col in zip(axes, num_cols):
        sns.boxplot(x=df[TARGET], y=df[col], ax=ax, palette='Set2')
        ax.set_title(f'{col} by {TARGET}')
    for ax in axes[len(num_cols):]:
        ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('TARGET is None or not categorical - skipped')

## Step 10: Interpretation Cheat-Sheet (write this in your exam)

| Observation | How to write it |
|---|---|
| Mean ~ Median, skew ~ 0 | Distribution is approximately symmetric / normal |
| Skewness > 0 (mean > median) | Right (positively) skewed; long tail on the right |
| Skewness < 0 (mean < median) | Left (negatively) skewed; long tail on the left |
| Kurtosis > 0 | Heavy tails / peaked (leptokurtic); < 0 is flat (platykurtic) |
| Points beyond whiskers in boxplot | Outliers (IQR rule: Q1 - 1.5 IQR, Q3 + 1.5 IQR) |
| \|r\| >= 0.7 | Strong correlation (possible multicollinearity) |
| 0.3 <= \|r\| < 0.7 | Moderate correlation |
| \|r\| < 0.3 | Weak or no linear relationship |
| r > 0 / r < 0 | Positive / negative linear relationship |

**Remember:** Correlation does not imply causation. Pearson captures only *linear* relationships.